# Day 33

In [7]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables from the project's .env file.
load_dotenv()

# Initialize Gemini through LangChain.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.3,
    max_retries=2
)

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [9]:
pdf_files = [
    "remote_work_policy.pdf",
    "business_travel_and_expenses.pdf"
]

documents = []

for pdf_file in pdf_files:

    if not Path(pdf_file).exists():
        raise FileNotFoundError(f"PDF not found: {pdf_file}")

    loader = PyPDFLoader(pdf_file)

    loaded_pages = loader.load()

    documents.extend(loaded_pages)

print("Total pages loaded:", len(documents))


Total pages loaded: 3


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print("Total chunks created:", len(chunks))

Total chunks created: 13


In [17]:
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='northstar_rag_demo',
    persist_directory='./doc.db'
)

print('Vector Store created successfully')

Vector Store created successfully


In [18]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [19]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a document question-answering assistant.

        Answer the question using ONLY the supplied context.

        If the answer is not available in the context,
        clearly state that the documents do not contain
        sufficient information.

        Do not invent facts.

        Context:
        {context}
        """
    ),
    ("human", "{question}")
])

In [20]:
def format_docs(docs):

    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        f"Page: {doc.metadata.get('page', 'Unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    )

In [21]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [22]:
question = "How many days must an employee work before requesting remote work?"

retrieved_docs = retriever.invoke(question)

print("\n========== RETRIEVED DOCUMENTS ==========\n")

for index, doc in enumerate(retrieved_docs, start=1):

    print(f"Document {index}")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Content:", doc.page_content)
    print("-" * 60)


========== RETRIEVED DOCUMENTS ==========

Document 1
Source: remote_work_policy.pdf
Page: 0
Content: 2. Eligibility and approval
An employee must complete at least 90 calendar days of employment before requesting a regular
remote-work arrangement. The employee's manager must approve the request in writing, and Human
Resources must record the approved arrangement.
Approval is role-dependent. Customer support roles that require on-site equipment may be limited to two
------------------------------------------------------------
Document 2
Source: remote_work_policy.pdf
Page: 0
Content: remote days per week. A manager may require additional office attendance for operational reasons.
3. Working hours and availability
The standard working day is eight working hours, excluding breaks. Employees must be reachable during
the team's agreed core hours of 10:00 a.m. to 3:00 p.m. in the team's primary time zone.
------------------------------------------------------------
Document 3
Source: remot

In [23]:
answer = rag_chain.invoke(question)

print("\n========== FINAL ANSWER ==========\n")
print(answer)


========== FINAL ANSWER ==========

Based on the provided document, an employee must complete at least 90 calendar days of employment before requesting a regular remote-work arrangement.
